In [ ]:
import os
# @title # Установка
# @markdown * Подключить гугл диск
mount_drive = False # @param {"type":"boolean"}
repo_url = "https://github.com/noblebarkrr/mvsepless"
mvsepless_dir = os.path.join("/content", "mvsepless-epsilon")
!git clone $repo_url -b epsilon $mvsepless_dir
!python mvsepless-epsilon/mvsepless/install.py
!python mvsepless-epsilon/mvsepless/separator.py info --update
if mount_drive:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
import os
from pyngrok import ngrok
import random
import string
import re
import urllib
import time
import ipywidgets as widgets
from IPython.display import display, Javascript
import threading
import subprocess

#@title # Web-UI
port = 7862
#@markdown * Способ поделится приложением
sharing_method = "gradio" # @param ["gradio","ngrok","localtunnel","not"]
#@markdown * Токен для ngrok *(где взять его - https://dashboard.ngrok.com/get-started/your-authtoken)*
ngrok_token = "" # @param {"type":"string"}
#@markdown * Включить Vbach в Web-UI
vbach = True # @param {"type":"boolean"}

lt_sub_domain = "mvsepless"
def generate_subdomain(length=8):
    """Генерация случайного субдомена заданной длины"""
    chars = string.ascii_lowercase + string.digits
    return ''.join(random.choice(chars) for _ in range(length))

if sharing_method == "ngrok":
    try:
        ngrok.set_auth_token(ngrok_token)
        ngrok.kill()
        tunnel = ngrok.connect(port)
        print(f"Публичная ссылка: {tunnel.public_url}")
    except KeyboardInterrupt:
        ngrok.kill()

if sharing_method == "localtunnel":
    os.system("npm install -g localtunnel &>/dev/null")
    time.sleep(7)
    with open('url.txt', 'w') as file:
        file.write('')
    subdomain = f"{re.sub(r'[^a-zA-Z0-9]', '', lt_sub_domain)}-{generate_subdomain(25)}"

    # Флаг для контроля работы потока
    tunnel_running = True

    def run_tunnel():
        while tunnel_running:
            print("localtunnel включается...")
            try:
                # Используем subprocess вместо os.system для лучшего контроля
                process = subprocess.Popen(
                    f'lt --port {port} '
                    f'{f"--subdomain {subdomain}" if lt_sub_domain != "" and not lt_sub_domain.isspace() else ""}',
                    shell=True,
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE
                )
                process.wait()  # Ждем завершения процесса
                if not tunnel_running:
                    break
                time.sleep(5)  # Пауза перед перезапуском
            except Exception as e:
                if tunnel_running:
                    print(f"Ошибка в localtunnel: {e}")
                    time.sleep(5)

    tunnel_thread = threading.Thread(target=run_tunnel, daemon=True)
    tunnel_thread.start()

    time.sleep(3)
    try:
        endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
        tunnel_url = f"https://{subdomain}.loca.lt"
        print(f"Публичная ссылка: {tunnel_url}")

        # Создаем текстовое поле с URL, а не IP
        text_field = widgets.Text(
            value=endpoint_ip,  # Исправлено: показываем URL, а не IP
            description='URL:',
            disabled=True
        )
        text_field.add_class("copy-enabled")

        display(text_field)

        # Исправленный JavaScript для копирования
        display(Javascript("""
        setTimeout(() => {
            const input = document.querySelector('.copy-enabled input');
            if (!input) return;

            const btn = document.createElement('button');
            btn.innerHTML = '📋';
            btn.style.cssText = `
                margin-left: 8px;
                border: none;
                background: none;
                cursor: pointer;
                font-size: 1.2em;
            `;
            input.parentNode.appendChild(btn);

            btn.addEventListener('click', () => {
                navigator.clipboard.writeText(input.value)  // Исправлено: input.value вместо input
                    .then(() => {
                        btn.innerHTML = '✓';
                        setTimeout(() => btn.innerHTML = '📋', 2000);
                    })
                    .catch(err => {
                        console.error('Ошибка копирования: ', err);
                    });
            });
        }, 300);
        """))

    except Exception as e:
        print(f"Ошибка при старте localtunnel: {e}")

    # Функция для корректного завершения
    def stop_tunnel():
        global tunnel_running
        tunnel_running = False
        print("Localtunnel завершает работу...")

    # Регистрируем обработчик для Ctrl+C
    import signal
    original_signal_handler = signal.getsignal(signal.SIGINT)

    def signal_handler(sig, frame):
        stop_tunnel()
        # Восстанавливаем оригинальный обработчик и вызываем его
        signal.signal(signal.SIGINT, original_signal_handler)
        raise KeyboardInterrupt

    signal.signal(signal.SIGINT, signal_handler)

cmd = ["python", "mvsepless-epsilon/mvsepless/separator.py", "app", "--port", str(port), "--add_app", "--use_plugins"]
if sharing_method == "gradio":
    cmd.append("--share")
if vbach:
    cmd.append("--vbach")

!{' '.join(cmd)}

# MVSepLess CLI

## Информация о моделях

In [ ]:
#@markdown * Фильтр по стему:
filter_stem = "" # @param {"type":"string","placeholder":"Vocals или Instrumental"}
#@markdown * Лимит:
limit = 0 # @param {"type":"integer"}
import shlex

cmd = ["python", "mvsepless-epsilon/mvsepless/separator.py", "info"]
if limit > 0:
    cmd.append("--limit")
    cmd.append(str(limit))
if filter_stem:
    cmd.append("--stem")
    cmd.append(filter_stem)

quoted_string = " ".join(shlex.quote(arg) for arg in cmd)
!{quoted_string}

## Инференс

In [ ]:
#@markdown ### Входные данные
#@markdown * Путь к входной папке/файлу:
input_path = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
#@markdown ---
#@markdown ### Выбор модели
#@markdown * Имя модели:
model_name = "bs_6stem" # @param ['mbr_vocals_kim', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_other_viperx', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'bandit_plus', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}

# @markdown ---
# @markdown ### Настройки разделения
# @markdown * Извлечь  инструментал:
instrumental = True # @param {type:"boolean"}
#@markdown ---
#@markdown ### Выходные данные
#@markdown * Формат:
output_format = "mp3" # @param ["mp3", "wav", "flac", "ogg", "opus", "m4a", "aac", "aiff"]
# @markdown * Битрейт
bitrate = 320 # @param {"type":"slider","min":32,"max":320,"step":1}
# @markdown * Выбрать выходные стемы(через запятую, например (vocal,  instrumental)):
stems_to_extract = "" # @param {type:"string"}
# @markdown * Шаблон именования выходных файлов:
output_template = "NAME (STEM) MODEL" # @param {type:"string"}
# @markdown * Длина чанка в минутах (для очень длинных аудио [отключить - 0]):
chunk_dur = 0 # @param {"type":"slider","min":0,"max":10,"step":0.1}
#@markdown * Путь к выходной папке:
output_dir = "/content/output2" # @param {"type":"string","placeholder":"/путь/к/папке"}

import shlex

cmd = [
    "python",
    "mvsepless-epsilon/mvsepless/separator.py",
    "--input", input_path,
    "--output_dir", output_dir,
    "--output_format", output_format,
    "--output_bitrate", str(bitrate),
    "separator",
    "--model_name", model_name,
    "--template", output_template
]

if chunk_dur > 0:
    cmd.extend(["--chunk_duration", str(chunk_dur * 60)])

if instrumental:
    cmd.append("--ext_inst")

if stems_to_extract:
    cmd.extend(["--selected_stems"] + stems_to_extract.split(","))

quoted_string = " ".join(shlex.quote(arg) for arg in cmd)
!{quoted_string}

## Ансамбль

### Авто-ансамбль (Максимум 10 моделей)

In [ ]:
#@markdown ### Внимание!
#@markdown * Для получения информации о стемах запустите ячейку в подразделе "Информация о моделях" раздела "MVSepless CLI"
#@markdown * В авто-ансамбле в инференсе по умолчанию выставлен параметр ext_inst
#@markdown ---
#@markdown ### Входные данные
#@markdown * Путь к входному файлу:
input_path = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
#@markdown ---
#@markdown ### Пресет
#@markdown ##### Описание параметров
#@markdown * model_name - имя модели
#@markdown * pri_stem - основной стем
#@markdown * sec_stem - инвертированный стем
#@markdown * weights - веса
#@markdown ---
#@markdown #### Модель 1
model_name1 = "" # @param ['mbr_vocals_kim', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_other_viperx', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'bandit_plus', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
pri_stem1 = "" # @param {"type":"string"}
sec_stem1 = "" # @param {"type":"string"}
weights1 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Модель 2
model_name2 = "" # @param ['mbr_vocals_kim', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_other_viperx', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'bandit_plus', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
pri_stem2 = "" # @param {"type":"string"}
sec_stem2 = "" # @param {"type":"string"}
weights2 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Модель 3
model_name3 = "" # @param ['mbr_vocals_kim', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_other_viperx', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'bandit_plus', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
pri_stem3 = "" # @param {"type":"string"}
sec_stem3 = "" # @param {"type":"string"}
weights3 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Модель 4
model_name4 = "" # @param ['mbr_vocals_kim', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_other_viperx', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'bandit_plus', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
pri_stem4 = "" # @param {"type":"string"}
sec_stem4 = "" # @param {"type":"string"}
weights4 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown ### Модель 5
model_name5 = "" # @param ['mbr_vocals_kim', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_other_viperx', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'bandit_plus', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
pri_stem5 = "" # @param {"type":"string"}
sec_stem5 = "" # @param {"type":"string"}
weights5 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Модель 6
model_name6 = "" # @param ['mbr_vocals_kim', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_other_viperx', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'bandit_plus', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
pri_stem6 = "" # @param {"type":"string"}
sec_stem6 = "" # @param {"type":"string"}
weights6 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Модель 7
model_name7 = "" # @param ['mbr_vocals_kim', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_other_viperx', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'bandit_plus', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
pri_stem7 = "" # @param {"type":"string"}
sec_stem7 = "" # @param {"type":"string"}
weights7 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Модель 8
model_name8 = "" # @param ['mbr_vocals_kim', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_other_viperx', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'bandit_plus', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
pri_stem8 = "" # @param {"type":"string"}
sec_stem8 = "" # @param {"type":"string"}
weights8 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Модель 9
model_name9 = "" # @param ['mbr_vocals_kim', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_other_viperx', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'bandit_plus', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
pri_stem9 = "" # @param {"type":"string"}
sec_stem9 = "" # @param {"type":"string"}
weights9 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Модель 10
model_name10 = "" # @param ['mbr_vocals_kim', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_other_viperx', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'bandit_plus', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
pri_stem10 = "" # @param {"type":"string"}
sec_stem10 = "" # @param {"type":"string"}
weights10 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown ### Выходные данные
#@markdown * Метод:
#@markdown > * min_fft - минимальная энергия
#@markdown > * max_fft - максимальная энергия
#@markdown > * avg_fft - взвешенное среднее (только с этим методом работают веса)
#@markdown > * median_fft - медиана
method = "min_fft" # @param ["min_fft", "max_fft", "avg_fft", "median_fft"]
#@markdown * Перевернуть ансамбль:
invert_ensemble = False # @param {"type":"boolean"}
#@markdown * Формат:
output_format = "mp3" # @param ["mp3", "wav", "flac", "ogg", "opus", "m4a", "aac", "aiff"]
#@markdown * Путь к выходной папке:
output_dir = "/content/ensemble_output" # @param {"type":"string","placeholder":"/путь/к/папке"}

import shlex

cmd = ["python", "mvsepless-epsilon/mvsepless/separator.py", "--input", input_path, "--output_format", output_format, "--output_dir", output_dir, "auto_ensemble", "--method", method]
if invert_ensemble:
    cmd.append("--invert")

cmd.append("--model_list")

if model_name1 != "":
    cmd.append(f"{model_name1},{pri_stem1},{sec_stem1},{weights1}")
if model_name2 != "":
    cmd.append(f"{model_name2},{pri_stem2},{sec_stem2},{weights2}")
if model_name3 != "":
    cmd.append(f"{model_name3},{pri_stem3},{sec_stem3},{weights3}")
if model_name4 != "":
    cmd.append(f"{model_name4},{pri_stem4},{sec_stem4},{weights4}")
if model_name5 != "":
    cmd.append(f"{model_name5},{pri_stem5},{sec_stem5},{weights5}")
if model_name6 != "":
    cmd.append(f"{model_name6},{pri_stem6},{sec_stem6},{weights6}")
if model_name7 != "":
    cmd.append(f"{model_name7},{pri_stem7},{sec_stem7},{weights7}")
if model_name8 != "":
    cmd.append(f"{model_name8},{pri_stem8},{sec_stem8},{weights8}")
if model_name9 != "":
    cmd.append(f"{model_name9},{pri_stem9},{sec_stem9},{weights9}")
if model_name10 != "":
    cmd.append(f"{model_name10},{pri_stem10},{sec_stem10},{weights10}")

quoted_string = " ".join(shlex.quote(arg) for arg in cmd)
!{quoted_string}

### Ручной ансамбль (Максимум 10 файлов)

In [ ]:
#@markdown ### Входные данные
#@markdown ---
#@markdown #### Файл 1
input_file1 = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
weights1 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Файл 2
input_file2 = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
weights2 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Файл 3
input_file3 = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
weights3 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Файл 4
input_file4 = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
weights4 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown ### Файл 5
input_file5 = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
weights5 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Файл 6
input_file6 = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
weights6 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Файл 7
input_file7 = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
weights7 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Файл 8
input_file8 = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
weights8 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Файл 9
input_file9 = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
weights9 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown #### Файл 10
input_file10 = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
weights10 = 1 # @param {"type":"number"}
#@markdown ---
#@markdown ### Выходные данные
#@markdown * Метод:
#@markdown > * min_fft - минимальная энергия
#@markdown > * max_fft - максимальная энергия
#@markdown > * avg_fft - взвешенное среднее (только с этим методом работают веса)
#@markdown > * median_fft - медиана
method = "max_fft" # @param ["min_fft", "max_fft", "avg_fft", "median_fft"]
#@markdown * Формат:
output_format = "mp3" # @param ["mp3", "wav", "flac", "ogg", "opus", "m4a", "aac", "aiff"]
#@markdown * Путь к выходной папке + имя выходного файла:
output_name = "/content/ensemble" # @param {"type":"string","placeholder":"/путь/к/файлу без расширения"}

import shlex

cmd = ["python", "mvsepless-epsilon/mvsepless/separator.py"]

files_args = ["--input"]
weights_args = ["--weights"]
if input_file1 != "":
    files_args.append(input_file1)
    weights_args.append(str(weights1))
if input_file2 != "":
    files_args.append(input_file2)
    weights_args.append(str(weights2))
if input_file3 != "":
    files_args.append(input_file3)
    weights_args.append(str(weights3))
if input_file4 != "":
    files_args.append(input_file4)
    weights_args.append(str(weights4))
if input_file5 != "":
    files_args.append(input_file5)
    weights_args.append(str(weights5))
if input_file6 != "":
    files_args.append(input_file6)
    weights_args.append(str(weights6))
if input_file7 != "":
    files_args.append(input_file7)
    weights_args.append(str(weights7))
if input_file8 != "":
    files_args.append(input_file8)
    weights_args.append(str(weights8))
if input_file9 != "":
    files_args.append(input_file9)
    weights_args.append(str(weights9))
if input_file10 != "":
    files_args.append(input_file10)
    weights_args.append(str(weights10))

cmd.extend(files_args)
cmd.extend(["--output_name", output_name, "--output_format", output_format, "manual_ensemble", "--method", method])
cmd.extend(weights_args)
quoted_string = " ".join(shlex.quote(arg) for arg in cmd)
!{quoted_string}

# VBach CLI

In [ ]:
#@title Показать список установленных моделей для преобразования
import shlex
cmd = ["python", "mvsepless-epsilon/mvsepless/vbach.py", "model_manager", "list"]

quoted_string = " ".join(shlex.quote(arg) for arg in cmd)
!{quoted_string}

In [ ]:
#@title Удаление голосовой модели
import shlex

voicemodel_name = "" # @param {"type":"string","placeholder":"Имя модели"}
cmd = ["python", "mvsepless-epsilon/mvsepless/vbach.py", "model_manager", "remove", "--model_name", voicemodel_name]

quoted_string = " ".join(shlex.quote(arg) for arg in cmd)
!{quoted_string}

## Установка голосовой модели

In [ ]:
#@title Через локальные файлы
import shlex
pth_path = "" # @param {"type":"string","placeholder":"Путь к *.pth файлу"}
index_path = "" # @param {"type":"string","placeholder":"Путь к *.index файлу"}
voicemodel_name = "" # @param {"type":"string","placeholder":"Имя модели"}
if pth_path != "" and voicemodel_name != "":
    cmd = ["python", "mvsepless-epsilon/mvsepless/vbach.py", "model_manager", "install_local", "--model_name", voicemodel_name, "--pth", pth_path]
    if index_path != "":
        cmd.extend(["--index", index_path])
    quoted_string = " ".join(shlex.quote(arg) for arg in cmd)
    !{quoted_string}

In [ ]:
#@title Через файлы с интернета
import shlex
pth_url = "" # @param {"type":"string","placeholder":"Ссылка на *.pth файл"}
index_url = "" # @param {"type":"string","placeholder":"Ссылка на *.index файл"}
voicemodel_name = "" # @param {"type":"string","placeholder":"Имя модели"}
if pth_url != "" and voicemodel_name != "":
    cmd = ["python", "mvsepless-epsilon/mvsepless/vbach.py", "model_manager", "install_url_files", "--model_name", voicemodel_name, "--pth_url", pth_url]
    if index_url != "":
        cmd.extend(["--index_url", index_url])
    quoted_string = " ".join(shlex.quote(arg) for arg in cmd)
    !{quoted_string}

In [ ]:
#@title Через zip файл с интернета
import shlex
zip_url = "" # @param {"type":"string","placeholder":"Ссылка на zip файл"}
voicemodel_name = "" # @param {"type":"string","placeholder":"Имя модели"}
if zip_url != "" and voicemodel_name != "":
    cmd = ["python", "mvsepless-epsilon/mvsepless/vbach.py", "model_manager", "install_url_zip", "--model_name", voicemodel_name, "--url", zip_url]
    quoted_string = " ".join(shlex.quote(arg) for arg in cmd)
    !{quoted_string}

## Инференс

In [ ]:
#@markdown ### Входные данные
#@markdown * Путь к входной папке/файлу:
input_path = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
#@markdown * Имя модели:
voicemodel_name = "" # @param {"type":"string","placeholder":"Имя модели"}
# @markdown ---
# @markdown  ### Hubert
# @markdown * Стэк
stack = "fairseq" # @param ["fairseq","transformers"]
# @markdown * Имя модели для fairseq
fairseq_embedder = "hubert_base" # @param ["hubert_base","contentvec_base","korean_hubert_base","chinese_hubert_base","portuguese_hubert_base","japanese_hubert_base"]
# @markdown * Имя модели для transformers
transformers_embedder = "contentvec" # @param ["contentvec","spin","spin-v2","chinese-hubert-base","japanese-hubert-base","korean-hubert-base"]
# @markdown ---
# @markdown  ### Настройки преобразования
# @markdown * Влияние индекса
index_rate = 1 # @param {"type":"slider","min":0,"max":1,"step":0.01}
# @markdown * Стерео режим
stereo_mode = "mono" # @param ["mono","left/right","sim/dif"]
# @markdown * Метод определения тона
method_pitch = "rmvpe+" # @param ["rmvpe+","mangio-crepe","mangio-crepe-tiny","fcpe",'harvest","pm","pyin"]
# @markdown * Изменение высоты тона (полутона)
pitch = 0 # @param {"type":"slider","min":-48,"max":48,"step":1}
# @markdown * Длина шага (для mangio-crepe)
hop_length = 128 # @param {"type":"slider","min":8,"max":512,"step":8}
# @markdown * Радиус фильтра
filter_radius = 3 # @param {"type":"slider","min":1,"max":7,"step":1}
# @markdown * Соотношение огибающих громкости
rms = 0.25 # @param {"type":"slider","min":0,"max":1,"step":0.01}
# @markdown * Защита согласных
protect = 0.33 # @param {"type":"slider","min":0,"max":0.5,"step":0.01}
# @markdown ---
#@markdown ### Дополнительные настройки
# @markdown * Минимальная частота F0
f0_min = 50 # @param {type:"integer"}
# @markdown * Максимальная частота F0
f0_max = 1100 # @param {type:"integer"}
# @markdown ---
#@markdown ### Выходные данные
#@markdown * Формат:
output_format = "mp3" # @param ["mp3", "wav", "flac", "ogg", "opus", "m4a", "aac", "aiff"]
# @markdown * Имя выходного файла:
output_name = "F0METHOD_PITCH_(MODEL)_NAME" # @param {type:"string"}
#@markdown * Путь к выходной папке:
output_dir = "/content/vbach_output" # @param {"type":"string","placeholder":"/путь/к/папке"}



import shlex
cmd = [
    "python",
    "mvsepless-epsilon/mvsepless/vbach.py", "cli",
    "--input", input_path,
    "--output_dir", output_dir,
    "--model_name", voicemodel_name,
    "--output_format", output_format,
    "--index_rate", str(index_rate),
    "--output_name", output_name,
    "--format_name",
    "--stereo_mode", stereo_mode,
    "--method_pitch", method_pitch,
    "--pitch", str(pitch),
    "--hop_length", str(hop_length),
    "--filter_radius", str(filter_radius),
    "--rms", str(rms),
    "--protect", str(protect),
    "--f0_min", str(f0_min),
    "--f0_max", str(f0_max),
    "--embedder_name",
    fairseq_embedder if stack == "fairseq" else transformers_embedder,
]
if stack == "transformers":
    cmd.append("--use_transformers")
quoted_string = " ".join(shlex.quote(arg) for arg in cmd)
!{quoted_string}

# Extra CLI

In [ ]:
#@title Скачивание аудио-файла с интернета
#@markdown ### Входные данные
#@markdown * Ссылка на аудио-файл:
url_to_audio = "" # @param {"type":"string","placeholder":"https://example.com/audio.mp3"}
#@markdown * Путь к cookies-файлу (если есть):
cookies_path = "" # @param {"type":"string","placeholder":"путь/к/файлу"}
#@markdown ---
#@markdown ### Выходные данные
#@markdown * Формат:
output_format = "mp3" # @param ["mp3", "wav", "flac", "ogg", "opus", "m4a", "aac", "aiff"]
# @markdown * Имя файла:
title = "" # @param {"type":"string","placeholder":"MVSEPLESS_AUDIO"}
#@markdown * Путь к выходной папке:
output_dir = "/content/output" # @param {"type":"string","placeholder":"/путь/к/папке"}
import shlex

cmd = ["python", "mvsepless-epsilon/mvsepless/gradio_helper.py", "--url", url_to_audio, "--output_dir", output_dir, "--output_format", output_format]
if title != "":
    cmd.extend(["--title", title])
if cookies_path != "":
    cmd.extend(["--cookie", cookies_path])
quoted_string = " ".join(shlex.quote(arg) for arg in cmd)
!{quoted_string}

In [ ]:
#@title Вычитание сигнала стема из оригинала

#@markdown ### Входные данные
#@markdown * Путь к оригинальному аудиофайлу:
original_audio_path = "" # @param {"type":"string","placeholder":"/путь/к/оригиналу.wav"}
#@markdown * Путь к файлу стема для вычитания:
stem_audio_path = "" # @param {"type":"string","placeholder":"/путь/к/стему.wav"}

#@markdown ---
#@markdown ### Настройки вычитания
#@markdown * Метод вычитания:
subtraction_method = "waveform" # @param ["waveform","spectrogram"]

#@markdown ---
#@markdown ### Выходные данные
#@markdown * Путь к выходному файлу:
output_inverter_path = "/content/inverted_output.wav" # @param {"type":"string","placeholder":"/путь/к/выходному/файлу.wav"}

import shlex

if original_audio_path and stem_audio_path and output_inverter_path:
    inverter_cmd = [
        "python",
        "mvsepless-epsilon/mvsepless/separator.py",
        "--input", original_audio_path,
        "subtract",
        "--stem", stem_audio_path,
        "--method", subtraction_method,
        "--output_path", output_inverter_path
    ]

    quoted_string = " ".join(shlex.quote(arg) for arg in cmd)
    !{quoted_string}


In [ ]:
#@title Разделение аудио на фантомный центр и стерео-базу

#@markdown ### Входные данные
#@markdown * Путь к входному файлу:
input_audio_path = "" # @param {"type":"string","placeholder":"/путь/к/входному/файлу.wav"}

#@markdown ---
#@markdown ### Выходные данные
#@markdown * Путь к выходному файлу центра:
output_path_mid = "/content/phantom_center_mid.wav" # @param {"type":"string","placeholder":"/путь/к/выходному/файлу_центра.wav"}
#@markdown * Путь к выходному файлу стерео-базы:
output_path_side = "/content/phantom_center_side.wav" # @param {"type":"string","placeholder":"/путь/к/выходному/файлу_стерео-базы.wav"}

import shlex

if input_audio_path and output_path_mid and output_path_side:
    cmd = [
        "python",
        "mvsepless/separator.py",
        "--input", input_audio_path,
        "ext_phantom_center",
        "--output_path_mid", output_path_mid,
        "--output_path_side", output_path_side
    ]

    quoted_string = " ".join(shlex.quote(arg) for arg in cmd)
    !{quoted_string}
